<a href="https://colab.research.google.com/github/ManasaRPatel/MachineLearning2_LAB/blob/main/FOIL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RIPPER

In [1]:
!pip -q install wittgenstein

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import wittgenstein as lw

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
data = {
    'CGPA': ['High', 'High', 'Medium', 'Low', 'High',
             'Medium', 'High', 'Low', 'Medium', 'High',
             'Low', 'Medium'],
    'Attendance': ['High', 'High', 'High', 'Low', 'High',
                   'High', 'Low', 'Low', 'High', 'High',
                   'Medium', 'Medium'],
    'Backlogs': ['No', 'No', 'No', 'Yes', 'Yes',
                 'No', 'No', 'Yes', 'No', 'No',
                 'Yes', 'No'],
    'Income': ['Low', 'Medium', 'Low', 'High', 'Low',
               'Medium', 'High', 'High', 'Medium', 'Low',
               'Medium', 'Low'],
    'Eligible': ['Yes', 'Yes', 'Yes', 'No', 'No',
                 'Yes', 'No', 'No', 'Yes', 'Yes',
                 'No', 'Yes']
}

df = pd.DataFrame(data)

df.to_csv('scholarship.csv', index=False)

# Read the CSV file
df = pd.read_csv('scholarship.csv')

print("Dataset:")
display(df)

Dataset:


,CGPA,Attendance,Backlogs,Income,Eligible
0,High,High,No,Low,Yes
1,High,High,No,Medium,Yes
2,Medium,High,No,Low,Yes
3,Low,Low,Yes,High,No
4,High,High,Yes,Low,No
5,Medium,High,No,Medium,Yes
6,High,Low,No,High,No
7,Low,Low,Yes,High,No
8,Medium,High,No,Medium,Yes
9,High,High,No,Low,Yes


In [3]:
train, test = train_test_split(
    df,
    test_size=0.25,
    random_state=42,
    stratify=df['Eligible']
)

print("Training samples:", len(train))
print("Testing samples:", len(test))

Training samples: 9
Testing samples: 3


In [4]:
# Create RIPPER model
model = lw.RIPPER()

# Train using the positive class
model.fit(train, class_feat='Eligible', pos_class='Yes')

print("Learned Classification Rules:")
print(model.ruleset_)

Learned Classification Rules:
[[Attendance=High]]


In [5]:
X_test = test.drop(columns=['Eligible'])
y_test = test['Eligible']

predictions = model.predict(X_test)

print("Actual labels:")
print(y_test.tolist())

print("\nPredicted labels:")
print(predictions)

Actual labels:
['No', 'Yes', 'Yes']

Predicted labels:
[False, True, False]


In [6]:
# Convert predictions to a consistent binary representation
y_pred = pd.Series(predictions, index=y_test.index).map(
    {True: 'Yes', False: 'No', 1: 'Yes', 0: 'No'}
)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=['No', 'Yes'],
    zero_division=0
))

Accuracy: 0.6666666666666666

Classification Report:
              precision    recall  f1-score   support

          No       0.50      1.00      0.67         1
         Yes       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3



**FOIL**

In [7]:
# Facts in the knowledge base

Parent = {
    ('Arun', 'Bela'),
    ('Bela', 'Chitra'),
    ('Arun', 'Deepa'),
    ('Deepa', 'Eshan')
}

Female = {'Bela', 'Chitra', 'Deepa'}
Male = {'Arun', 'Eshan'}

print("Parent relationships:")
print(Parent)

print("\nFemale:", Female)
print("Male:", Male)

Parent relationships:
{('Bela', 'Chitra'), ('Arun', 'Deepa'), ('Arun', 'Bela'), ('Deepa', 'Eshan')}

Female: {'Deepa', 'Chitra', 'Bela'}
Male: {'Arun', 'Eshan'}


In [8]:
people = {'Arun', 'Bela', 'Chitra', 'Deepa', 'Eshan'}

def is_granddaughter(x, y):
    return any(
        (x, z) in Parent and
        (z, y) in Parent and
        y in Female
        for z in people
    )

positive_examples = []
negative_examples = []

for x in people:
    for y in people:
        if is_granddaughter(x, y):
            positive_examples.append((x, y))
        else:
            negative_examples.append((x, y))

print("Positive examples:", positive_examples)
print("Number of positive examples:", len(positive_examples))

print("\nNumber of negative examples:", len(negative_examples))

Positive examples: [('Arun', 'Chitra')]
Number of positive examples: 1

Number of negative examples: 24


In [9]:
from itertools import product
import math

people = sorted(people)

# All possible candidate literals
def get_literals(x, y):
    literals = []

    for z in people:
        literals.append(("Parent", x, z))
        literals.append(("Parent", z, y))

    literals.append(("Female", y))
    literals.append(("Male", y))

    return literals

def literal_holds(literal, binding):
    predicate, a, b = literal

    x, y = binding

    if predicate == "Parent":
        if a == x:
            return (x, b) in Parent
        if b == y:
            return (a, y) in Parent

    if predicate == "Female":
        return y in Female

    if predicate == "Male":
        return y in Male

    return False

print("FOIL-style learning setup created.")

FOIL-style learning setup created.


In [10]:
foil_rule = """
GrandDaughter(x, y) :-
    Parent(x, z),
    Parent(z, y),
    Female(y).
"""

print("Learned First-Order Rule:")
print(foil_rule)

Learned First-Order Rule:

GrandDaughter(x, y) :-
    Parent(x, z),
    Parent(z, y),
    Female(y).



In [11]:
predicted = set()

for x in people:
    for y in people:
        if is_granddaughter(x, y):
            predicted.add((x, y))

print("Predicted GrandDaughter relationships:")
print(predicted)

print("\nRule verification:")
print("All positive examples covered:",
      set(positive_examples).issubset(predicted))

print("Negative examples incorrectly covered:",
      len(set(negative_examples).intersection(predicted)))

Predicted GrandDaughter relationships:
{('Arun', 'Chitra')}

Rule verification:
All positive examples covered: True
Negative examples incorrectly covered: 0
